# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Raqib-Usman/ML_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

# Load data
csv_files = list(Path(".").rglob("content_refresh_anonymized.csv"))
if not csv_files:
    import subprocess, os
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/M-Raqib-Usman/ML_Internship.git", "repo_temp"], check=True)
    os.chdir("repo_temp")
    csv_files = list(Path(".").rglob("content_refresh_anonymized.csv"))

df = pd.read_csv(csv_files[0])
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print("Loaded:", df.shape)

Loaded: (30000, 45)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 – “Random Forest achieves Precision@50 ≈ 0.74 vs baseline ≈ 0.24 on a client-holdout split.”**

Methodology question I would ask:  
Where exactly does the declining label come from, and is the same definition used for both the model and the baseline?  
Is the client-holdout truly sealed (no pages from test clients appear in training feature engineering)?  
This is a strong ranking claim; confirming the label definition and the exact split code would make it even more trustworthy.

**Finding 2 – “The model roughly triples the hand-written rule at the top of the queue.”**

Methodology question I would ask:  
Is Precision@50 the only metric, or do Precision@20 and Precision@100 tell the same story?  
Does the lift survive when the baseline is also given access to the same features the model sees?  
Reporting the full ranking curve (not just one K) would strengthen the claim further.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I compare a **random row split** (optimistic) against a **client-holdout split** (honest).  
The gap between them shows how much the model was memorizing client-specific patterns.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = [
    "content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "word_count", "engagement_rate", "sessions_90d"
]
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values
groups = df["client_id"].values

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- 1. Random split (before) ---
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
rf_random = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
rf_random.fit(X_tr_r, y_tr_r)
scores_random = rf_random.predict_proba(X_te_r)[:, 1]

# --- 2. Client-holdout split (after / honest) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
X_tr_g, X_te_g = X.iloc[tr_idx], X.iloc[te_idx]
y_tr_g, y_te_g = y[tr_idx], y[te_idx]

rf_group = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
rf_group.fit(X_tr_g, y_tr_g)
scores_group = rf_group.predict_proba(X_te_g)[:, 1]

# --- Comparison ---
comparison = pd.DataFrame({
    "Split": ["Random row split", "Client-holdout (honest)"],
    "Precision@50": [
        precision_at_k(scores_random, y_te_r, 50),
        precision_at_k(scores_group, y_te_g, 50)
    ],
    "ROC-AUC": [
        roc_auc_score(y_te_r, scores_random),
        roc_auc_score(y_te_g, scores_group)
    ],
    "Base rate": [y_te_r.mean(), y_te_g.mean()]
}).round(3)

print(comparison.to_string(index=False))
print("\nThe gap (if any) shows how much optimistic leakage the random split allowed.")

                  Split  Precision@50  ROC-AUC  Base rate
       Random row split          0.86    0.746      0.542
Client-holdout (honest)          0.64    0.596      0.511

The gap (if any) shows how much optimistic leakage the random split allowed.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Features used ===")
print(feature_cols)

print("\n=== Leakage checklist ===")
print("1. Label-derived columns present in features?",
      any(c in feature_cols for c in ["trend_pct", "trend_direction", "is_declining_label"]))
print("2. Product flags / health scores used?",
      any("health" in c.lower() or "priority" in c.lower() for c in feature_cols))
print("3. Future-window columns used?",
      any("last_30" in c or "trend" in c for c in feature_cols))

# Deliberate leak test (then remove)
print("\n=== Deliberate leak test (should jump toward 1.0) ===")
X_leaky = X.copy()
X_leaky["leaky_trend"] = df["trend_pct"].fillna(0)

tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X_leaky, y, groups))
rf_leaky = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_leaky.fit(X_leaky.iloc[tr_idx], y[tr_idx])
leaky_scores = rf_leaky.predict_proba(X_leaky.iloc[te_idx])[:, 1]
print(f"Precision@50 WITH leaky feature: {precision_at_k(leaky_scores, y[te_idx], 50):.3f}")
print("→ Score jumps when we leak. Confirms our audit harness works. We do NOT keep the leaky feature.")

=== Features used ===
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'engagement_rate', 'sessions_90d']

=== Leakage checklist ===
1. Label-derived columns present in features? False
2. Product flags / health scores used? False
3. Future-window columns used? False

=== Deliberate leak test (should jump toward 1.0) ===
Precision@50 WITH leaky feature: 1.000
→ Score jumps when we leak. Confirms our audit harness works. We do NOT keep the leaky feature.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language.*

**Original (too strong):**  
“The Random Forest model is 3× better than the rule and should be used to decide which pages to refresh.”

**Rewritten (honest):**  
“On a client-holdout split of this anonymized dataset, the Random Forest ranked declining pages higher than a transparent hand-written rule (Precision@50 improved from ~0.24–0.40 range to the 0.6–0.75 range depending on run).  
The output is **decision-support only**: it surfaces pages that look higher-priority for human review.  
It does not prove that refreshing those pages will recover traffic, and it does not claim to reverse-engineer any search algorithm.”

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.